# Diamond EDA Overview

This overview was created to organize the existing CSV versions and reports. It is
not an original notebook recovered from yesterday. The three recovered original notebooks are directly in `sideEDA/`;
additional snapshots are in `history/`; the maintained experiments
remain in `notebooks/02_classification.ipynb`.

The current classifier starts from `data/raw/diamonds.csv`. Earlier EDA cleaned CSVs
are kept separately and are not silently substituted as its input.


In [ ]:
from pathlib import Path
import json
import hashlib
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
                    if (folder / "src" / "classification" / "pipeline.py").is_file())
EDA_DIR = PROJECT_ROOT / "sideEDA"


## CSV Versions and Current Classification Input

These counts identify file versions; they do not imply identical cleaning rules.


In [ ]:
versions = {
    "Raw classification input": PROJECT_ROOT / "data" / "diamonds.csv",
    "Earlier EDA cleaned": EDA_DIR / "data" / "cleaned" / "diamonds_cleaned.csv",
    "Earlier EDA final cleaned": EDA_DIR / "data" / "cleaned" / "diamonds_final_cleaned.csv",
    "Current classification cleaned export": PROJECT_ROOT / "data" / "classification" / "cleaned" / "dataset.csv"
}
version_frames = {name: pd.read_csv(path) for name, path in versions.items()}
version_summary = pd.DataFrame([
    {"Version": name, "Rows": len(version_frames[name]), "Columns": len(version_frames[name].columns),
     "Path": str(path.relative_to(PROJECT_ROOT)), "SHA256": hashlib.sha256(path.read_bytes()).hexdigest()}
    for name, path in versions.items()
])
display(version_summary)


## DataFrames, Types, Missing Values, and Statistics


In [ ]:
for name, df in version_frames.items():
    print(name)
    display(df)
    df.info()
    display(df.isna().sum().rename("Missing values").to_frame())
    display(df.describe(include="all").T)


## Earlier Cleaning Diagnostics

These preserved CSVs contain flagged rows. The recovered notebooks contain the earlier exploration code.


In [ ]:
for path in sorted((EDA_DIR / "data" / "diagnostics").glob("*.csv")):
    df = pd.read_csv(path)
    print(path.name, df.shape)
    display(df)
    display(df.describe(include="all").T)


## Historical Experiment Reports

These are historical reports, separate from the three current classification experiments.


In [ ]:
for path in sorted((EDA_DIR / "outputs" / "reports").rglob("*.csv")):
    print(path.name)
    display(pd.read_csv(path))


## Current Classification Data: Distributions


In [ ]:
df = version_frames["Current classification cleaned export"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df["Clarity_Family"].value_counts().reindex(["I", "SI", "VS", "VVS", "IF"]).plot.bar(ax=axes[0], rot=0)
axes[0].set(title="Clarity families", xlabel="Clarity", ylabel="Rows")
df["carat"].plot.hist(ax=axes[1], bins=40)
axes[1].set(title="Carat distribution", xlabel="Carat")
df["price"].plot.hist(ax=axes[2], bins=40)
axes[2].set(title="Price distribution", xlabel="Price")
plt.tight_layout()
plt.show()


## File Inventory and Snapshot Verification

A changed hash may mean an active dataset was regenerated since this inventory was created.


In [ ]:
inventory = json.loads((EDA_DIR / "manifest.json").read_text(encoding="utf-8"))
checks = []
for item in inventory:
    path = PROJECT_ROOT / item["path"]
    exists = path.is_file()
    checks.append({"File": item["path"], "Exists": exists,
                   "Matches recorded SHA256": exists and hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"]})
display(pd.DataFrame(checks))
